# AgTech MRV — Demo de Reportes Satelitales para Certificación ARC/AAPRESID

Este notebook demuestra el pipeline completo de procesamiento satelital para generar reportes de evidencia de **siembra directa** y **cobertura vegetal permanente**, utilizando imágenes Sentinel-2 (óptico) y Sentinel-1 (SAR).

El sistema calcula métricas relativas (lote vs. departamento) para auto-calibrarse sin intervención de un agrónomo, y genera un PDF de 2 páginas apto para adjuntar al expediente de certificación ARC de AAPRESID.

**Datos:** Sentinel-2 L2A + Sentinel-1 SAR GRD  
**Zona de prueba:** Lote agrícola cerca de Casilda, Santa Fe (pampa húmeda)

In [ ]:
# Setup: importar dependencias e inicializar GEE
%matplotlib inline

import ee
import json
import urllib.request
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from PIL import Image

from gee_processor import (
    inicializar_gee,
    cargar_poligono,
    cargar_sentinel2,
    cargar_sentinel1,
    calcular_metricas_lote,
    serie_temporal_ndvi,
    serie_temporal_nbr,
    detectar_caidas_abruptas,
    detectar_eventos_quema,
    calcular_dnbr,
    comparacion_departamental,
    calcular_confianza,
    diagnosticar_siembra_directa,
    procesar_lote,
    obtener_imagen_rgb,
    evaluar_disponibilidad_datos,
)
from pdf_generator import generar_pdf

# Inicializar Google Earth Engine
inicializar_gee(proyecto='your-gee-project')

In [ ]:
# Definir polígono de prueba y parámetros del análisis
GEOJSON_PATH = 'sample_polygons/lote_santa_fe.geojson'
FECHA_INICIO = '2023-10-01'
FECHA_FIN = '2024-04-30'

# Cargar polígono
with open(GEOJSON_PATH, 'r') as f:
    geojson_data = json.load(f)

print(f"Nombre: {geojson_data['properties']['nombre']}")
print(f"Superficie aprox: {geojson_data['properties']['superficie_aprox_ha']} ha")
print(f"Coordenadas del pol\u00edgono:")
for coord in geojson_data['geometry']['coordinates'][0]:
    print(f"  lon={coord[0]:.3f}, lat={coord[1]:.3f}")

# Cargar como geometría GEE
poligono = cargar_poligono(GEOJSON_PATH)
superficie_m2 = poligono.area().getInfo()
superficie_ha = round(superficie_m2 / 10000, 2)
print(f"\nSuperficie calculada por GEE: {superficie_ha} ha")

## Carga de Datos Satelitales

Cargamos las colecciones Sentinel-2 (imágenes ópticas multiespectrales) y Sentinel-1 (radar SAR) para el período de análisis. Se aplica filtrado de nubes automático.

In [ ]:
# Cargar colección Sentinel-2 y mostrar imagen RGB del lote
s2_col = cargar_sentinel2(poligono, FECHA_INICIO, FECHA_FIN)
s1_col = cargar_sentinel1(poligono, FECHA_INICIO, FECHA_FIN)

# Generar thumbnail RGB (mediana temporal)
rgb_url = obtener_imagen_rgb(s2_col, poligono)

# Descargar y mostrar imagen RGB
req = urllib.request.Request(rgb_url, headers={'User-Agent': 'AgTech-MRV/1.0'})
with urllib.request.urlopen(req, timeout=30) as response:
    img_data = response.read()

img = Image.open(BytesIO(img_data))

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(img)
ax.set_title('Imagen RGB — Mediana temporal Sentinel-2', fontsize=14)
ax.set_xlabel(f'Per\u00edodo: {FECHA_INICIO} a {FECHA_FIN}')
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## Índices Espectrales

Calculamos los índices espectrales clave para evaluar siembra directa:

- **NDVI**: Cobertura vegetal (mayor = más vegetación)
- **EVI**: Vegetación (mejorado, corrige por atmósfera y suelo)
- **BSI**: Suelo desnudo (mayor = más suelo expuesto, posible labranza)
- **NBR**: Quema (valores negativos = posible quema de rastrojo)
- **RVI**: Rugosidad SAR (bajo = suelo liso/labrado, alto = no labrado)

In [ ]:
# Calcular métricas espectrales del lote
metricas = calcular_metricas_lote(s2_col, s1_col, poligono)

print("\n" + "=" * 45)
print("  M\u00e9tricas del Lote (mediana temporal)")
print("=" * 45)
print(f"  NDVI medio:  {metricas['ndvi_medio']:.4f}")
print(f"  NDVI m\u00edn:   {metricas['ndvi_min']:.4f}")
print(f"  EVI medio:   {metricas['evi_medio']:.4f}")
print(f"  BSI medio:   {metricas['bsi_medio']:.4f}")
print(f"  NBR medio:   {metricas['nbr_medio']:.4f}")
print(f"  NBR m\u00edn:    {metricas['nbr_min']:.4f}")
sar_label = f"{metricas['rvi_medio']:.4f}" if metricas['rvi_disponible'] else 'Sin datos SAR'
print(f"  RVI medio:   {sar_label}")
print("=" * 45)

In [ ]:
# Serie temporal NDVI — evolución de la cobertura vegetal
serie_ndvi = serie_temporal_ndvi(s2_col, poligono)

# Preparar datos para el gráfico
fechas = [datetime.strptime(p['fecha'], '%Y-%m-%d') for p in serie_ndvi]
valores_ndvi = [p['ndvi'] for p in serie_ndvi]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fechas, valores_ndvi, 'o-', color='#2d8a4e', linewidth=1.5,
        markersize=4, label='NDVI medio del lote')
ax.fill_between(fechas, valores_ndvi, alpha=0.15, color='#2d8a4e')
ax.axhline(y=0.2, color='#c0392b', linestyle='--', alpha=0.5,
           label='Umbral m\u00ednimo cobertura (0.20)')
ax.set_title('Serie Temporal NDVI — Cobertura Vegetal del Lote', fontsize=14)
ax.set_ylabel('NDVI')
ax.set_xlabel('Fecha')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.set_ylim(-0.1, 1.0)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Detectar ca\u00eddas abruptas (posible labranza)
caidas = detectar_caidas_abruptas(serie_ndvi)

In [ ]:
# Detección de quema de rastrojo (serie NBR + dNBR)
serie_nbr = serie_temporal_nbr(s2_col, poligono)
eventos_quema = detectar_eventos_quema(serie_nbr)
dnbr_eventos = calcular_dnbr(serie_nbr, eventos_quema)

# Gráfico de serie NBR
fechas_nbr = [datetime.strptime(p['fecha'], '%Y-%m-%d') for p in serie_nbr]
valores_nbr = [p['nbr'] for p in serie_nbr]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fechas_nbr, valores_nbr, 'o-', color='#e67e22', linewidth=1.5,
        markersize=4, label='NBR medio del lote')
ax.fill_between(fechas_nbr, valores_nbr, alpha=0.15, color='#e67e22')
ax.axhline(y=0, color='#c0392b', linestyle='--', alpha=0.5,
           label='L\u00ednea de referencia (NBR = 0)')
ax.set_title('Serie Temporal NBR — Detecci\u00f3n de Quema de Rastrojo', fontsize=14)
ax.set_ylabel('NBR')
ax.set_xlabel('Fecha')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Mostrar resultados de quema
if eventos_quema:
    print("\nEventos de quema detectados:")
    for ev in eventos_quema:
        print(f"  Fecha: {ev['fecha']}, NBR: {ev['nbr']:.4f}, "
              f"Severidad: {ev['severidad']}, Duraci\u00f3n: {ev['duracion_obs']} obs")
    if dnbr_eventos:
        print("\nAn\u00e1lisis dNBR (metodolog\u00eda USGS):")
        for d in dnbr_eventos:
            print(f"  {d['fecha']}: dNBR={d['dnbr']:.4f} "
                  f"(pre={d['nbr_pre']:.4f}, post={d['nbr_post']:.4f}) "
                  f"-> Severidad: {d['severidad_usgs']}")
else:
    print("\nSin eventos de quema detectados en el per\u00edodo analizado.")

## Comparación Departamental

El sistema calcula las métricas del **departamento** (FAO GAUL nivel 2) donde se ubica el lote y compara mediante z-scores. Esto permite auto-calibrar el análisis sin umbrales absolutos: un lote con NDVI por encima de la media departamental tiene mejor cobertura que sus vecinos.

In [ ]:
# Comparación del lote contra el departamento
comparacion = comparacion_departamental(poligono, FECHA_INICIO, FECHA_FIN, metricas)

print("\n" + "=" * 55)
print(f"  Comparaci\u00f3n: Lote vs. {comparacion['nombre_departamento']}, {comparacion['provincia']}")
print("=" * 55)
print(f"  NDVI lote:         {metricas['ndvi_medio']:.4f}")
print(f"  NDVI departamento: {comparacion['ndvi_departamental']:.4f} "
      f"(\u00b1{comparacion['ndvi_depto_stddev']:.4f})")
print(f"  NDVI z-score:      {comparacion['ndvi_z_score']:+.2f}")
print(f"  BSI z-score:       {comparacion['bsi_z_score']:+.2f}")
print(f"  NBR departamento:  {comparacion['nbr_departamental']:.4f}")
print()

# Interpretación del z-score NDVI
z = comparacion['ndvi_z_score']
if z >= 0.5:
    interp = "Muy por encima de la media — excelente cobertura relativa"
elif z >= 0:
    interp = "Sobre la media departamental — buena cobertura"
elif z >= -0.5:
    interp = "Levemente bajo la media — cobertura aceptable"
elif z >= -1.0:
    interp = "Significativamente bajo la media — revisar"
else:
    interp = "Muy por debajo de la media — cobertura deficiente"

print(f"  Interpretaci\u00f3n: {interp}")

# Interpretación del z-score BSI
bsi_z = comparacion['bsi_z_score']
if bsi_z <= 0:
    bsi_interp = "BSI bajo la media — bajo suelo desnudo (positivo)"
elif bsi_z <= 1.0:
    bsi_interp = "BSI en rango normal del departamento"
else:
    bsi_interp = "BSI alto respecto al departamento — posible labranza"

print(f"  BSI:             {bsi_interp}")

## Diagnóstico Final

El sistema evalúa tres preguntas convergentes:
1. **¿Hubo labranza?** (RVI + caídas abruptas + BSI)
2. **¿Hay cobertura permanente?** (NDVI mín relativo + posición zonal)
3. **¿Hubo quema de rastrojo?** (NBR serie temporal + dNBR)

Y calcula un score de confianza (0-100) que determina si el reporte puede emitirse sin revisión humana.

In [ ]:
# Ejecutar diagnóstico completo
resultado = procesar_lote(GEOJSON_PATH, FECHA_INICIO, FECHA_FIN,
                          proyecto_gee='your-gee-project')

diagnostico = resultado['diagnostico']
confianza = diagnostico['confianza']

# Mostrar criterios evaluados
print("\n" + "=" * 60)
print("  DIAGN\u00d3STICO DE SIEMBRA DIRECTA")
print("=" * 60)

for criterio in diagnostico['criterios']:
    print(f"\n  {criterio['pregunta']} -> {criterio['resultado']}")
    for ev in criterio['evidencia']:
        print(f"    - {ev}")

# Score de confianza
print("\n" + "-" * 60)
print(f"  Score de confianza: {confianza['score']}/100 ({confianza['categoria']})")
print(f"  Requiere revisi\u00f3n: {'S\u00ed' if confianza['requiere_revision'] else 'No'}")
print("-" * 60)

print("\n  Desglose del score:")
for comp in confianza['detalle']:
    print(f"    {comp['componente']}: {comp['puntaje']}/25 — {comp['nota']}")

# Conclusión
print("\n" + "=" * 60)
estado = 'CONSISTENTE CON SIEMBRA DIRECTA' if diagnostico['consistente'] \
    else 'REQUIERE VERIFICACI\u00d3N'
print(f"  RESULTADO: {estado}")
print("=" * 60)
print(f"\n  {diagnostico['conclusion']}")

In [ ]:
# Generar reporte PDF para certificación ARC/AAPRESID
import os
os.makedirs('sample_outputs', exist_ok=True)

output_path = 'sample_outputs/reporte_mrv_demo.pdf'
ruta_pdf = generar_pdf(resultado, output_path=output_path)
print(f"\nPDF generado exitosamente: {ruta_pdf}")
print(f"Tama\u00f1o: {os.path.getsize(ruta_pdf) / 1024:.1f} KB")

---

**Reporte generado exitosamente.** El PDF se encuentra en `sample_outputs/`.

Este reporte de 2 páginas incluye mapas RGB y NDVI del lote, gráfico temporal, tabla de métricas y diagnóstico automático, listo para adjuntar al expediente de certificación ARC de AAPRESID.